# 76 — Campaign C staged M5（child M4 完走後）

前提: notebook 75 で child M4 が `M4_COMPLETE` / enclosure `BLOCKED_MATH`。

## なぜまだ走らせるか（q&lt;1）

- Campaign C screening は **q≈1.01 付近のヒューリスティック**であり、q≥1 の証明ではない
- M3 influence≈1、M4 は `missing_bound_terms` が多く **決定的上界が無い**
- したがって **q&lt;1 も q≥1 も未排除**。M5 は義務評価を fail-closed で記録する探索段階

## 非主張

- 常に `NOT_CERTIFIED`
- open obligations が残る限り `ONE_STEP_CERTIFIED` は出さない
- 連続極限・質量ギャップは主張しない


In [ ]:
from pathlib import Path
import os
import sys

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'm5_orchestrator.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/m5_orchestrator.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault(
    'VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT',
)
os.environ.setdefault('VALIDATED_RG_M7C_RUN_ID', 'M7-20260720T081500Z-c8d5f02b3c96')
os.environ.setdefault('VALIDATED_RG_STAGED_CANDIDATE', 'CAND-000005-ac85c5e9ba38')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)


In [ ]:
import json
from pathlib import Path

from src.common import read_json

M7C_RUN_ID = os.environ.get('VALIDATED_RG_M7C_RUN_ID', 'M7-20260720T081500Z-c8d5f02b3c96')
DEFAULT_CAND = os.environ.get('VALIDATED_RG_STAGED_CANDIDATE', 'CAND-000005-ac85c5e9ba38')
explicit_pkg = os.environ.get('VALIDATED_RG_STAGED_PACKAGE')
PACKAGE_ROOT = (
    Path(explicit_pkg).expanduser().resolve()
    if explicit_pkg else
    (PERSIST_ROOT / 'searches' / M7C_RUN_ID / 'auto_execute' / DEFAULT_CAND).resolve()
)
required = ['scheme.json', 'resource_gate.json', 'child_run_ids.json']
missing = [n for n in required if not (PACKAGE_ROOT / n).is_file()]
if missing:
    raise FileNotFoundError(f'Package incomplete: {PACKAGE_ROOT} missing {missing}')

CHILD_IDS = read_json(PACKAGE_ROOT / 'child_run_ids.json')
M4_RUN_ID = CHILD_IDS['M4']
M5_RUN_ID = CHILD_IDS['M5']
os.environ['VALIDATED_RG_M4_RUN_ID'] = M4_RUN_ID
os.environ['VALIDATED_RG_M5_RUN_ID'] = M5_RUN_ID

m4_acceptance = PERSIST_ROOT / 'runs' / M4_RUN_ID / 'reports' / 'M4_acceptance.json'
m4_report = PERSIST_ROOT / 'runs' / M4_RUN_ID / 'reports' / 'M4_report.json'
if not m4_acceptance.is_file() or not m4_report.is_file():
    raise FileNotFoundError(
        f'Child M4 not complete under {M4_RUN_ID}. Finish notebook 75 first.'
    )
print('PACKAGE_ROOT', PACKAGE_ROOT)
print('M4_RUN_ID', M4_RUN_ID)
print('M5_RUN_ID', M5_RUN_ID)


## M4→M5 audit rewrite + parent verify


In [ ]:
from src.m5_config import default_m5_config
from src.m5_parent import verify_accepted_m4_parent
from src.m7_staged_lineage import write_child_m4_acceptance_audit

audit_path = PROJECT_ROOT / 'audit' / 'm4_accepted_parent.json'
audit = read_json(audit_path) if audit_path.is_file() else None
if not isinstance(audit, dict) or audit.get('accepted_run_id') != M4_RUN_ID:
    print(
        'Rewriting audit/m4_accepted_parent.json from child M4:',
        M4_RUN_ID,
        '(was', None if not isinstance(audit, dict) else audit.get('accepted_run_id'), ')',
    )
    audit = write_child_m4_acceptance_audit(
        PROJECT_ROOT,
        run_root=PERSIST_ROOT / 'runs' / M4_RUN_ID,
    )
if not isinstance(audit, dict) or audit.get('accepted_run_id') != M4_RUN_ID:
    raise RuntimeError('M4 audit rewrite failed to pin child run.')

m4_over = {}
m4_over_path = PACKAGE_ROOT / 'm4_config_overrides.json'
if m4_over_path.is_file():
    m4_over = read_json(m4_over_path)
projected = int(m4_over.get('projected_rank') or 16)
scheme = read_json(PACKAGE_ROOT / 'scheme.json')
j2_max = int(scheme.get('j2_max') or 2)

M5_CONFIG = default_m5_config(
    parent_m4_run_id=M4_RUN_ID,
    run_id=M5_RUN_ID,
    cutoff=j2_max,
    bond_dimension=projected,
    mode='staged_child',
)
PARENT_EVIDENCE = verify_accepted_m4_parent(PROJECT_ROOT, PERSIST_ROOT, M4_RUN_ID)
print(json.dumps({
    'parent_run': M4_RUN_ID,
    'm5_run': M5_RUN_ID,
    'mode': M5_CONFIG.mode,
    'cutoff': M5_CONFIG.cutoff,
    'bond_dimension': M5_CONFIG.bond_dimension,
    'checkpoint_index': audit.get('checkpoint_index'),
    'enclosure_status': audit.get('enclosure_status'),
    'milestone_status_audit': audit.get('milestone_status'),
    'open_for_M5': PARENT_EVIDENCE.bound_ledger.get('open_for_M5'),
    'tensor_count': len(PARENT_EVIDENCE.tensors),
    'certification_status': 'NOT_CERTIFIED',
}, indent=2, ensure_ascii=False, default=str))


## M5 セッション（義務評価 → 必要なら live assembly）

open obligations が残れば結果は `BLOCKED_MATH` / `NOT_CERTIFIED` のまま（正常）。


In [ ]:
from src.m5_orchestrator import create_or_resume_m5
from src.m5_status import NOT_CERTIFIED, ONE_STEP_CERTIFIED
from src.runtime import environment_info, validate_persistent_root

PERSIST = validate_persistent_root()
print(json.dumps(environment_info(), ensure_ascii=False, indent=2, default=str))

m5_orchestrator = create_or_resume_m5(
    PERSIST,
    M5_CONFIG,
    PROJECT_ROOT,
    run_id=M5_RUN_ID,
)
print('run_id', m5_orchestrator.config.run_id)
print('parent', m5_orchestrator.config.parent_m4_run_id)

M5_RESULT = m5_orchestrator.run_until_checkpoint()
# Staged child keeps open M4→M5 obligations; ONE_STEP_CERTIFIED is not expected.
assert M5_RESULT.get('certification_status') == NOT_CERTIFIED
assert M5_RESULT.get('certification_status') != ONE_STEP_CERTIFIED
print(json.dumps(M5_RESULT, indent=2, ensure_ascii=False, default=str))


## 結果確認


In [ ]:
from src.common import read_json

reports = m5_orchestrator.run_root / 'reports'
summary = {
    'run_id': m5_orchestrator.config.run_id,
    'parent_m4': m5_orchestrator.config.parent_m4_run_id,
    'reports_dir': str(reports),
}
for name in (
    'M5_report.json', 'M5_acceptance.json', 'M5_obligation_report.json',
    'M5_parent_artifact_inventory.json',
):
    path = reports / name
    summary[name] = path.is_file()
print(json.dumps(summary, indent=2, ensure_ascii=False))

obl = reports / 'M5_obligation_report.json'
if obl.is_file():
    doc = read_json(obl)
    print(json.dumps({
        '判定': 'M5 obligation evaluation present (exploratory; NOT_CERTIFIED unless all closed).',
        'all_closed': doc.get('all_closed'),
        'closed_obligations': doc.get('closed_obligations'),
        'open_obligations': doc.get('open_obligations'),
        'certification_status': M5_RESULT.get('certification_status'),
        'milestone_status': M5_RESULT.get('milestone_status'),
        'enclosure_status': M5_RESULT.get('enclosure_status'),
    }, indent=2, ensure_ascii=False, default=str))
else:
    print('M5 obligation report missing — inspect M5_RESULT / logs.')
